# Model — Xception — Sunflower Alternaria Blight Severity Classification

Six-class severity classifier (Immune, Resistant, MR, MS, S, HS — Mayee & Datar 1986 scale).
Transfer learning from ImageNet, class-imbalance handling, full performance + efficiency metrics.

**Run order:** Sections 1–2 must be run identically on all 8 systems (same `split.csv`, same seed).
Only Section 4 (model definition) changes between the 8 notebooks.

| Setting | Value |
|---|---|
| Model | Xception (timm `legacy_xception`, ImageNet weights, new `fc` head) |
| Input size | 299×299 |
| Early stopping patience | 25 (on val macro-F1) |
| Batch size | 32 |
| Optimizer | AdamW |
| LR schedule | Two-stage: frozen head warmup, then full fine-tune w/ cosine decay |


## 0. Setup

In [1]:
# One-time environment check (run this in a terminal, not as a Python cell):
# python -c "import timm, thop, sklearn, matplotlib, seaborn, pandas, tqdm; print('all present')"

In [ ]:
!pip install -q timm thop scikit-learn matplotlib seaborn pandas tqdm
import os, time, json, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, roc_auc_score, cohen_kappa_score,
                              classification_report)
import matplotlib.pyplot as plt
import seaborn as sns
from thop import profile as thop_profile
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # tolerate partially-truncated JPEGs instead of crashing
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


## 1. Dataset access (Google Drive)

Fill in **one** of the two options below.

- **Colab + Drive mount** (simplest if you're on Colab and the data is in your own Drive)
- **gdown** (works anywhere, incl. local/Kaggle, if the folder/file is shared "Anyone with the link")

Paste your actual link/ID once you have it — code is ready either way, just uncomment the block you need.


In [3]:
# Dataset is already local -- no Drive download needed.
DATA_ROOT = "/home/sem_7th/Downloads/Sunflower"

# Expected layout (ImageFolder-style, one subfolder per class):
# DATA_ROOT/
#   Class 1/  *.jpg
#   Class 2/  *.jpg
#   ...
#   Class 6/  *.jpg

CLASS_NAMES = ["Class 1", "Class 2", "Class 3", "Class 4", "Class 5", "Class 6"]
SEVERITY_SCORE = {0: 0, 1: 1, 2: 3, 3: 5, 4: 7, 5: 9}          # label idx -> 0-9 score
SEVERITY_LABEL = {0: "Immune", 1: "Resistant", 2: "MR", 3: "MS", 4: "S", 5: "HS"}

## 2. Stratified 70/15/15 split — GENERATE ONCE, REUSE ON ALL 8 SYSTEMS

Run this cell only the **first** time. It writes `split.csv` (path, label, split).
Copy that exact file to the other 7 systems/notebooks and load it instead of
regenerating — this guarantees every one of the 8 models is trained/evaluated
on identical data, which is required for the comparison in Section 4 of the report.


In [4]:
SPLIT_CSV = "split.csv"


def is_valid_image_file(path):
    try:
        from PIL import Image
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


if not os.path.exists(DATA_ROOT):
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}. Update it to your Google Drive folder path.")

if not os.path.exists(SPLIT_CSV):
    records = []
    missing_dirs = []
    skipped = 0
    for label_idx, cname in enumerate(CLASS_NAMES):
        class_dir = os.path.join(DATA_ROOT, cname)
        if not os.path.isdir(class_dir):
            missing_dirs.append(class_dir)
            continue
        for fname in sorted(os.listdir(class_dir)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            fpath = os.path.join(class_dir, fname)
            if not is_valid_image_file(fpath):
                skipped += 1
                print(f"Skipping unreadable image: {fpath}")
                continue
            records.append({"path": fpath, "label": label_idx})

    if not records:
        raise FileNotFoundError(
            "No valid images were found. Check that DATA_ROOT and CLASS_NAMES match your Drive folder structure. "
            f"Expected folders: {[os.path.join(DATA_ROOT, c) for c in CLASS_NAMES]}"
        )

    if missing_dirs:
        print("Missing class folders:")
        for d in missing_dirs:
            print(" -", d)
    if skipped:
        print(f"Skipped unreadable files: {skipped}")

    df = pd.DataFrame(records)
    print("Total valid images found:", len(df))
    print(df["label"].value_counts().sort_index())

    train_df, rest_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
    val_df, test_df = train_test_split(rest_df, test_size=0.50, stratify=rest_df["label"], random_state=SEED)

    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"
    full = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
    full.to_csv(SPLIT_CSV, index=False)
    print(f"Saved {SPLIT_CSV} -- COPY THIS FILE TO ALL 8 SYSTEMS AND REUSE IT.")
else:
    print(f"{SPLIT_CSV} already exists -- loading existing split (do NOT regenerate).")

split_df = pd.read_csv(SPLIT_CSV)
valid_mask = split_df["path"].apply(is_valid_image_file)
split_df = split_df[valid_mask].reset_index(drop=True)
print(f"Kept {len(split_df)} valid rows after filtering unreadable files.")
print(split_df.groupby(["split", "label"]).size().unstack(fill_value=0))


split.csv already exists -- loading existing split (do NOT regenerate).
Kept 5540 valid rows after filtering unreadable files.
label    0    1    2     3    4    5
split                               
test   140   89  160   218  157   67
train  650  415  751  1017  730  315
val    139   89  161   218  156   68


## 3. Dataset / DataLoaders + class-imbalance handling

In [5]:
IMG_SIZE = 299
# Xception's pretrained weights (Keras/Cadene port) expect inputs scaled to [-1, 1]
# i.e. mean=std=0.5 -- NOT the ImageNet mean/std used by the other notebooks.
# (Set USE_XCEPTION_NORM = False to fall back to ImageNet stats if you want a strictly identical pipeline.)
USE_XCEPTION_NORM = True
NORM_MEAN = [0.5, 0.5, 0.5] if USE_XCEPTION_NORM else [0.485, 0.456, 0.406]
NORM_STD  = [0.5, 0.5, 0.5]   if USE_XCEPTION_NORM else [0.229, 0.224, 0.225]

# Phone-camera-aware augmentation for the TRAIN set only
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),        # leaves have no fixed "up" -- biologically fine
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.02),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.15),  # phone motion/focus blur
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08)),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

class LeafDataset(Dataset):
    def __init__(self, df, transform):
        self.paths = df["path"].tolist()
        self.labels = df["label"].tolist()
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        from PIL import Image
        path = self.paths[idx]
        try:
            with Image.open(path) as img:
                img = img.convert("RGB")
                img = self.transform(img)
            return img, self.labels[idx]
        except Exception as e:
            raise ValueError(f"Unreadable image file: {path}") from e

train_ds = LeafDataset(split_df[split_df.split == "train"], train_tf)
val_ds   = LeafDataset(split_df[split_df.split == "val"], eval_tf)
test_ds  = LeafDataset(split_df[split_df.split == "test"], eval_tf)

# --- class imbalance: balanced batch sampling for train loader ---
train_labels = split_df[split_df.split == "train"]["label"].values
class_counts = np.bincount(train_labels, minlength=len(CLASS_NAMES))
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print("Train class counts:", class_counts)
print("Train/Val/Test sizes:", len(train_ds), len(val_ds), len(test_ds))

# Also compute per-class weights for an (optional) weighted loss term.
# We use the sampler as the primary imbalance fix and keep the loss unweighted
# + light label smoothing, to avoid double-correcting. Flip WEIGHTED_LOSS=True
# to additionally weight CE if the sampler alone isn't enough on your data.
WEIGHTED_LOSS = False
loss_class_weights = torch.tensor(
    class_weights / class_weights.sum() * len(CLASS_NAMES),
    dtype=torch.float32
).to(DEVICE)

Train class counts: [ 650  415  751 1017  730  315]
Train/Val/Test sizes: 3878 831 831


## 4. Model — Xception (transfer learning)

In [ ]:
MODEL_NAME = "Xception"

def build_model(num_classes=6, dropout=0.3):
    import timm
    # torchvision has no Xception, so it comes from timm. In timm >= 0.9 the original (Keras/Cadene-port)
    # Xception is called "legacy_xception"; in older timm versions it is called "xception".
    # `drop_rate` is applied right before the final `fc` layer (same position/rate as the other notebooks).
    try:
        m = timm.create_model("legacy_xception", pretrained=True, num_classes=num_classes, drop_rate=dropout)
    except Exception:
        m = timm.create_model("xception", pretrained=True, num_classes=num_classes, drop_rate=dropout)
    return m

model = build_model(len(CLASS_NAMES)).to(DEVICE)

def set_backbone_trainable(model, trainable: bool):
    # Xception's classification head is `fc` (everything else is backbone)
    for name, p in model.named_parameters():
        if not name.startswith("fc."):
            p.requires_grad = trainable


NameError: name 'timm' is not defined

## 5. Training config

**Two-stage fine-tuning:**
1. Freeze backbone, train the new head only (fast, stabilizes the randomly-initialized classifier).
2. Unfreeze everything, fine-tune end-to-end at a much lower LR.

Early stopping (patience = 25, monitored on **validation macro-F1**, not just loss/accuracy,
since macro-F1 is what actually reflects performance on the minority classes) applies to stage 2.


In [ ]:
STAGE1_EPOCHS = 10          # head-only warmup
STAGE2_MAX_EPOCHS = 150     # capped upper bound; early stopping will normally trigger sooner
PATIENCE = 25
LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05


criterion = nn.CrossEntropyLoss(
    weight=loss_class_weights if WEIGHTED_LOSS else None,
    label_smoothing=LABEL_SMOOTHING
)

CHECKPOINT_PATH = f"best_{MODEL_NAME}.pth"


## 6. Train / eval loop helpers

In [ ]:
def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(train_mode):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_preds.append(outputs.argmax(1).detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
    all_preds = np.concatenate(all_preds); all_labels = np.concatenate(all_labels)
    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)[2]
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, macro_f1


def train_stage(model, optimizer, scheduler, n_epochs, patience, history, start_epoch=0):
    best_f1 = -1
    epochs_no_improve = 0
    for epoch in range(start_epoch, start_epoch + n_epochs):
        t0 = time.time()
        train_loss, train_acc, train_f1 = run_epoch(model, train_loader, optimizer)
        val_loss, val_acc, val_f1 = run_epoch(model, val_loader, optimizer=None)
        epoch_time = time.time() - t0
        if scheduler is not None:
            scheduler.step(val_f1)

        history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc, train_f1=train_f1,
                             val_loss=val_loss, val_acc=val_acc, val_f1=val_f1, epoch_time_s=epoch_time))
        print(f"[{epoch:03d}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_f1={train_f1:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f} | {epoch_time:.1f}s")

        if val_f1 > best_f1:
            best_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), CHECKPOINT_PATH)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch} (no val_f1 improvement for {patience} epochs).")
                break
    return epoch + 1  # last epoch index reached


## 7. Run training — Stage 1 (head only) → Stage 2 (full fine-tune, early stopping patience=25)

In [ ]:
history = []

# --- Stage 1: frozen backbone, head warmup ---
set_backbone_trainable(model, trainable=False)
optimizer1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=LR_STAGE1, weight_decay=WEIGHT_DECAY)
print("=== Stage 1: training head only ===")
last_epoch = train_stage(model, optimizer1, scheduler=None, n_epochs=STAGE1_EPOCHS,
                          patience=STAGE1_EPOCHS + 1, history=history, start_epoch=0)

# --- Stage 2: unfreeze everything, fine-tune with early stopping ---
set_backbone_trainable(model, trainable=True)
optimizer2 = torch.optim.AdamW(model.parameters(), lr=LR_STAGE2, weight_decay=WEIGHT_DECAY)
scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode="max", factor=0.5, patience=8)
print("=== Stage 2: full fine-tune (early stopping patience=25 on val macro-F1) ===")
train_stage(model, optimizer2, scheduler2, n_epochs=STAGE2_MAX_EPOCHS, patience=PATIENCE,
            history=history, start_epoch=last_epoch)

# reload best checkpoint before evaluation
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
print("Loaded best checkpoint for final evaluation.")


@torch.no_grad()
def evaluate_on_split(model, loader, name):
    model.eval()
    all_logits, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = probs.argmax(1)
    acc = accuracy_score(labels, preds)
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    print(f"{name.upper()} ACC={acc:.4f} | Macro-P={prec_macro:.4f} | Macro-R={rec_macro:.4f} | Macro-F1={f1_macro:.4f}")
    return acc, f1_macro


evaluate_on_split(model, test_loader, "test")


## 8. Learning curves

In [ ]:
hist_df = pd.DataFrame(history)
hist_df.to_csv(f"history_{MODEL_NAME}.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df.epoch, hist_df.train_loss, label="train")
axes[0].plot(hist_df.epoch, hist_df.val_loss, label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(hist_df.epoch, hist_df.train_f1, label="train")
axes[1].plot(hist_df.epoch, hist_df.val_f1, label="val")
axes[1].set_title("Macro-F1"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(f"learning_curves_{MODEL_NAME}.png", dpi=150)
plt.show()


## 9. Test-set performance metrics (Section 3 of the report)

In [ ]:
@torch.no_grad()
def get_test_predictions(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_logits), np.concatenate(all_labels)

logits, y_true = get_test_predictions(model, test_loader)
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
y_pred = probs.argmax(1)

acc = accuracy_score(y_true, y_pred)
prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
kappa = cohen_kappa_score(y_true, y_pred, weights="quadratic")  # ordinal severity -> quadratic weighting

try:
    auc_ovr = roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
except ValueError:
    auc_ovr = float("nan")

cm = confusion_matrix(y_true, y_pred)
# per-class specificity from the confusion matrix
specificities = []
for i in range(len(CLASS_NAMES)):
    tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
    fp = cm[:, i].sum() - cm[i, i]
    specificities.append(tn / (tn + fp + 1e-9))
specificity_macro = float(np.mean(specificities))

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
print(f"Accuracy: {acc:.4f}")
print(f"Macro  P/R/F1: {prec_macro:.4f} / {rec_macro:.4f} / {f1_macro:.4f}")
print(f"Weighted P/R/F1: {prec_w:.4f} / {rec_w:.4f} / {f1_w:.4f}")
print(f"Specificity (macro): {specificity_macro:.4f}")
print(f"AUC-ROC (OvR, macro): {auc_ovr:.4f}")
print(f"Cohen's kappa (quadratic-weighted): {kappa:.4f}")

# confusion matrix heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"{MODEL_NAME} -- Confusion Matrix (test set)")
plt.tight_layout()
plt.savefig(f"confusion_matrix_{MODEL_NAME}.png", dpi=150)
plt.show()


## 9b. Misclassified images — CSV + class-wise image folders

Exports every **test-set** image the model got wrong:

- `misclassified_<MODEL_NAME>/misclassified_test.csv` — one row per wrong image (path, true/pred class, confidence, probability of the true class, 2nd choice, severity gap)
- `misclassified_<MODEL_NAME>/<True_Class>/predicted_as_<Pred_Class>/` — copies of the wrong images, grouped by their **true class**, then by what the model confused them with. The file name carries the predicted class and confidence.
- `misclassification_summary_per_class.csv` and `misclassification_pairs.csv` — error rate per class and the most frequent true→predicted confusions

Images are **copied**, never moved, so your dataset stays untouched.

In [ ]:
import shutil
from PIL import Image


def _safe(name):
    return str(name).replace(" ", "_").replace("/", "-")


def export_misclassified(ds, y_true, y_pred, probs, out_dir, split_name="test", copy_images=True):
    """Save misclassified samples to CSV and (optionally) into <out_dir>/<True_Class>/predicted_as_<Pred_Class>/."""
    os.makedirs(out_dir, exist_ok=True)
    paths = ds.paths
    wrong_idx = np.where(np.asarray(y_true) != np.asarray(y_pred))[0]

    rows = []
    for i in wrong_idx:
        t, p = int(y_true[i]), int(y_pred[i])
        order = np.argsort(-probs[i])
        second = int(order[1])
        src = paths[i]
        stem, ext = os.path.splitext(os.path.basename(src))
        conf = float(probs[i, p])

        saved_path = ""
        if copy_images:
            dst_dir = os.path.join(out_dir, _safe(CLASS_NAMES[t]), f"predicted_as_{_safe(CLASS_NAMES[p])}")
            os.makedirs(dst_dir, exist_ok=True)
            dst = os.path.join(dst_dir, f"{stem}__pred-{_safe(CLASS_NAMES[p])}__conf-{conf:.2f}{ext}")
            if os.path.exists(dst):                       # same file name from different folders
                dst = os.path.join(dst_dir, f"{stem}__{i}__pred-{_safe(CLASS_NAMES[p])}__conf-{conf:.2f}{ext}")
            try:
                shutil.copy2(src, dst)
                saved_path = dst
            except Exception as e:
                print(f"Could not copy {src}: {e}")

        rows.append({
            "split": split_name,
            "image_name": os.path.basename(src),
            "source_path": src,
            "saved_path": saved_path,
            "true_label": t,
            "true_class": CLASS_NAMES[t],
            "true_severity": SEVERITY_LABEL[t],
            "pred_label": p,
            "pred_class": CLASS_NAMES[p],
            "pred_severity": SEVERITY_LABEL[p],
            "pred_confidence": round(conf, 4),
            "prob_of_true_class": round(float(probs[i, t]), 4),
            "second_choice_class": CLASS_NAMES[second],
            "severity_gap_classes": abs(t - p),              # 1 = neighbouring grade, 3+ = serious miss
            "severity_score_error": abs(SEVERITY_SCORE[t] - SEVERITY_SCORE[p]),
        })

    mis_df = pd.DataFrame(rows)
    if len(mis_df):
        mis_df = mis_df.sort_values(["true_label", "pred_confidence"], ascending=[True, False]).reset_index(drop=True)
    csv_path = os.path.join(out_dir, f"misclassified_{split_name}.csv")
    mis_df.to_csv(csv_path, index=False)

    # per-class summary
    summ = []
    for c in range(len(CLASS_NAMES)):
        n_total = int((np.asarray(y_true) == c).sum())
        sub = mis_df[mis_df.true_label == c] if len(mis_df) else mis_df
        top_conf = sub.pred_class.value_counts().idxmax() if len(sub) else "-"
        summ.append({"class": CLASS_NAMES[c], "severity": SEVERITY_LABEL[c], "n_images": n_total,
                     "n_misclassified": len(sub),
                     "error_rate": round(len(sub) / max(n_total, 1), 4),
                     "most_common_wrong_prediction": top_conf})
    summ_df = pd.DataFrame(summ)
    summ_df.to_csv(os.path.join(out_dir, "misclassification_summary_per_class.csv"), index=False)

    if len(mis_df):
        pairs_df = (mis_df.groupby(["true_class", "pred_class"]).size()
                    .reset_index(name="count").sort_values("count", ascending=False))
    else:
        pairs_df = pd.DataFrame(columns=["true_class", "pred_class", "count"])
    pairs_df.to_csv(os.path.join(out_dir, "misclassification_pairs.csv"), index=False)

    print(f"[{split_name}] {len(mis_df)} / {len(y_true)} misclassified "
          f"({len(mis_df) / max(len(y_true), 1):.1%}).  CSV -> {csv_path}")
    return mis_df, summ_df, pairs_df


MIS_DIR = f"misclassified_{MODEL_NAME}"
mis_df, mis_summary_df, mis_pairs_df = export_misclassified(
    test_ds, y_true, y_pred, probs, out_dir=MIS_DIR, split_name="test", copy_images=True)

display(mis_summary_df)
print("\nTop confusions (true -> predicted):")
display(mis_pairs_df.head(10))
print("\nHow far off are the mistakes? (severity_gap_classes)")
print(mis_df["severity_gap_classes"].value_counts().sort_index().to_string() if len(mis_df) else "none")

# --- preview: the most CONFIDENT mistakes (usually mislabeled images or genuinely ambiguous leaves) ---
N_PREVIEW = 12
if len(mis_df):
    top = mis_df.sort_values("pred_confidence", ascending=False).head(N_PREVIEW)
    cols = 4
    rows_n = int(np.ceil(len(top) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(3.4 * cols, 3.6 * rows_n))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (_, r) in zip(axes, top.iterrows()):
        try:
            ax.imshow(Image.open(r.source_path).convert("RGB"))
        except Exception:
            pass
        ax.set_title(f"true: {r.true_class}\npred: {r.pred_class} ({r.pred_confidence:.2f})", fontsize=9, color="crimson")
    plt.suptitle(f"{MODEL_NAME} -- most confident misclassifications (test set)", y=1.0)
    plt.tight_layout()
    plt.savefig(os.path.join(MIS_DIR, "top_confident_errors.png"), dpi=130, bbox_inches="tight")
    plt.show()

# Optional: do the same for the validation set (set True if you also want to inspect val errors)
EXPORT_VAL_ERRORS = False
if EXPORT_VAL_ERRORS:
    val_logits, val_true = get_test_predictions(model, val_loader)
    val_probs = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
    export_misclassified(val_ds, val_true, val_probs.argmax(1), val_probs,
                         out_dir=f"misclassified_{MODEL_NAME}_val", split_name="val")

## 10. Efficiency metrics — params, model size, FLOPs, training time, inference latency

In [ ]:
num_params = sum(p.numel() for p in model.parameters())
num_params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

torch.save(model.state_dict(), f"final_{MODEL_NAME}.pth")
model_size_mb = os.path.getsize(f"final_{MODEL_NAME}.pth") / (1024 ** 2)

dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
flops, _ = thop_profile(model, inputs=(dummy_input,), verbose=False)
gflops = flops / 1e9

avg_epoch_time_s = hist_df["epoch_time_s"].mean()

# --- inference latency: batch=1, average of 100 runs after 10 warmup runs ---
def measure_latency(model, device, n_warmup=10, n_runs=100):
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_runs):
            _ = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.time()
    return (t1 - t0) / n_runs * 1000  # ms/image

latency_gpu_ms = measure_latency(model, DEVICE) if DEVICE.type == "cuda" else None
latency_cpu_ms = measure_latency(model.to("cpu"), torch.device("cpu"))
model.to(DEVICE)  # move back

print(f"Params (total / trainable): {num_params:,} / {num_params_trainable:,}")
print(f"Model size (.pth): {model_size_mb:.2f} MB")
print(f"FLOPs: {gflops:.2f} GFLOPs")
print(f"Avg training time/epoch: {avg_epoch_time_s:.1f} s")
print(f"Inference latency -- GPU: {latency_gpu_ms} ms/img | CPU: {latency_cpu_ms:.2f} ms/img")


## 11. Save this model's results row

This appends one row to `results_master.csv`. Once all 8 models (run across your 8 systems)
have appended their row to a **shared copy** of this file, that CSV *is* the Section 4
comparison table — feed it straight into the trade-off plots (accuracy/F1 vs. latency vs. size).


In [ ]:
result_row = {
    "model": MODEL_NAME,
    "accuracy": acc,
    "precision_macro": prec_macro,
    "recall_macro": rec_macro,
    "f1_macro": f1_macro,
    "precision_weighted": prec_w,
    "recall_weighted": rec_w,
    "f1_weighted": f1_w,
    "specificity_macro": specificity_macro,
    "auc_roc_ovr_macro": auc_ovr,
    "cohen_kappa_quadratic": kappa,
    "params_total": num_params,
    "params_trainable_stage2": num_params_trainable,
    "model_size_mb": model_size_mb,
    "gflops": gflops,
    "avg_train_time_per_epoch_s": avg_epoch_time_s,
    "epochs_run": int(hist_df.epoch.max()) + 1,
    "inference_latency_gpu_ms": latency_gpu_ms,
    "inference_latency_cpu_ms": latency_cpu_ms,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
}

MASTER_CSV = "results_master.csv"
row_df = pd.DataFrame([result_row])
if os.path.exists(MASTER_CSV):
    row_df.to_csv(MASTER_CSV, mode="a", header=False, index=False)
else:
    row_df.to_csv(MASTER_CSV, index=False)

print(row_df.T)
print(f"\nAppended to {MASTER_CSV}. Copy/merge this file across your 8 systems to build the full comparison table.")


## 12. Ablation study — hyperparameter search to improve accuracy

The baseline (test metrics in Section 9) uses a very small fine-tuning LR (`1e-5`) and a fixed recipe. This section measures **which hyperparameters actually matter** and then retrains the best combination with the full budget.

**How it works**

1. **One-factor-at-a-time (greedy) ablation.** Starting from the baseline config, each factor is swept while the others stay at their current best. The best value is kept only if it beats the current best by at least `MIN_GAIN` validation macro-F1 (guards against picking noise). Factors, in order: fine-tune LR → LR scheduler → image size → augmentation → dropout → weight decay → label smoothing → batch size → imbalance handling.
2. **Short budget per run** (`ABL_STAGE1` + `ABL_STAGE2` epochs, short patience) so the whole sweep is affordable. Runs are **ranked on the validation set only** — the test set is never used for choosing hyperparameters.
3. **Final retrain** of the winning config with the full baseline budget (head warm-up + up to 150 epochs, patience 25), evaluated once on the test set and compared with the baseline.
4. Everything is logged to `ablation_results.csv` after each run, and the sweep **resumes automatically** if the kernel stops (set `RESUME_ABLATION = False` to start over).

**Outputs:** `ablation_results.csv`, `ablation_summary.png`, `best_config.json`, `best_Xception_tuned.pth`, `baseline_vs_tuned.csv`, `misclassified_Xception_tuned/`.

> ⏱ Time: each ablation run ≈ `ABL_STAGE1 + ABL_STAGE2` epochs (larger image sizes are slower). The default search space is ~18 runs. Shrink `SEARCH_SPACE` or the epoch budget if you need it faster. A single seed on a ~720-image validation set has roughly ±1% noise, so treat gains below ~1% as inconclusive.

In [ ]:
import copy, json, itertools

USE_AMP = DEVICE.type == "cuda"      # mixed precision: faster ablation runs on GPU, no effect on CPU

# snapshot of the baseline's test metrics (from Section 9) for the final comparison
baseline_test = {"accuracy": float(acc), "precision_macro": float(prec_macro), "recall_macro": float(rec_macro),
                 "f1_macro": float(f1_macro), "kappa_quadratic": float(kappa), "auc_roc_ovr_macro": float(auc_ovr)}

# ---------------- search configuration ----------------
BASE_CFG = dict(
    lr_stage1=LR_STAGE1,
    lr_stage2=LR_STAGE2,
    scheduler="plateau",           # "plateau" (ReduceLROnPlateau) | "cosine"
    img_size=IMG_SIZE,
    aug="baseline",                # "light" | "baseline" | "strong"
    dropout=0.3,
    weight_decay=WEIGHT_DECAY,
    label_smoothing=LABEL_SMOOTHING,
    batch_size=BATCH_SIZE,
    imbalance="sampler",           # "sampler" | "sampler+loss" | "loss_only"
)

# Ordered: most influential factors first. Edit freely -- the baseline value is cached, so it is never re-run.
SEARCH_SPACE = {
    "lr_stage2":       [1e-5, 3e-5, 1e-4, 3e-4],
    "scheduler":       ["plateau", "cosine"],
    "img_size":        [299, 352],
    "aug":             ["light", "baseline", "strong"],
    "dropout":         [0.2, 0.3, 0.5],
    "weight_decay":    [1e-4, 1e-3, 1e-2],
    "label_smoothing": [0.0, 0.05, 0.1],
    "batch_size":      [16, 32, 64],
    "imbalance":       ["sampler", "sampler+loss", "loss_only"],
}

# budget for each ablation run (short) -- the final retrain uses the full baseline budget
ABL_STAGE1, ABL_STAGE2, ABL_PATIENCE = 3, 20, 6
MIN_GAIN = 0.005                     # a new value must beat the current best val macro-F1 by this much
RESUME_ABLATION = True
ABLATION_CSV = "ablation_results.csv"


# ---------------- builders ----------------
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def build_transforms(img_size, aug):
    norm = transforms.Normalize(NORM_MEAN, NORM_STD)
    eval_t = transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor(), norm])
    if aug == "light":
        train_t = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.9, 1.0)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.ToTensor(), norm])
    elif aug == "strong":
        train_t = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.3),
            transforms.RandomRotation(30),
            transforms.RandAugment(num_ops=2, magnitude=7),
            transforms.ToTensor(), norm,
            transforms.RandomErasing(p=0.25, scale=(0.02, 0.12))])
    else:  # "baseline" = same pipeline as Section 3
        train_t = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.3),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.02),
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.15),
            transforms.ToTensor(), norm,
            transforms.RandomErasing(p=0.15, scale=(0.02, 0.08))])
    return train_t, eval_t


def make_loaders(cfg):
    train_t, eval_t = build_transforms(cfg["img_size"], cfg["aug"])
    tr_df = split_df[split_df.split == "train"]
    tr_ds = LeafDataset(tr_df, train_t)
    va_ds = LeafDataset(split_df[split_df.split == "val"], eval_t)
    te_ds = LeafDataset(split_df[split_df.split == "test"], eval_t)

    labels = tr_df["label"].values
    counts = np.bincount(labels, minlength=len(CLASS_NAMES))
    cw = 1.0 / np.maximum(counts, 1)
    use_sampler = cfg["imbalance"].startswith("sampler")
    sampler = WeightedRandomSampler(cw[labels], num_samples=len(labels), replacement=True) if use_sampler else None
    bs = cfg["batch_size"]
    tr_ld = DataLoader(tr_ds, batch_size=bs, sampler=sampler, shuffle=not use_sampler,
                       num_workers=4, pin_memory=True, drop_last=True)   # drop_last: avoids a 1-image BN batch
    va_ld = DataLoader(va_ds, batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)
    te_ld = DataLoader(te_ds, batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)
    loss_w = torch.tensor(cw / cw.sum() * len(CLASS_NAMES), dtype=torch.float32)
    return tr_ld, va_ld, te_ld, te_ds, loss_w


def build_model_cfg(dropout):
    try:
        return timm.create_model("legacy_xception", pretrained=True, num_classes=len(CLASS_NAMES), drop_rate=dropout)
    except Exception:
        return timm.create_model("xception", pretrained=True, num_classes=len(CLASS_NAMES), drop_rate=dropout)


In [ ]:
def _run_one_epoch(m, loader, crit, optimizer=None, scaler=None):
    train_mode = optimizer is not None
    m.train() if train_mode else m.eval()
    total_loss, preds, labs = 0.0, [], []
    with torch.set_grad_enabled(train_mode):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out = m(x)
                loss = crit(out, y)
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            preds.append(out.argmax(1).cpu().numpy()); labs.append(y.cpu().numpy())
    preds, labs = np.concatenate(preds), np.concatenate(labs)
    f1 = precision_recall_fscore_support(labs, preds, average="macro", zero_division=0)[2]
    return total_loss / len(loader.dataset), accuracy_score(labs, preds), f1


def train_with_config(cfg, stage1_epochs, stage2_epochs, patience, tag="run", verbose=False, keep_model=False):
    """Two-stage training (head warm-up -> full fine-tune) with the given hyperparameters.
    Model selection is on validation macro-F1. Returns a result dict (+ model/loaders if keep_model)."""
    seed_everything(SEED)
    tr_ld, va_ld, te_ld, te_ds, loss_w = make_loaders(cfg)
    m = build_model_cfg(cfg["dropout"]).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=loss_w.to(DEVICE) if "loss" in cfg["imbalance"] else None,
                               label_smoothing=cfg["label_smoothing"])
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

    hist, best = [], {"f1": -1, "acc": 0, "epoch": -1, "state": None}
    t_start = time.time()

    def _record(ep, stage, tr, va):
        hist.append(dict(epoch=ep, stage=stage, train_loss=tr[0], train_acc=tr[1], train_f1=tr[2],
                         val_loss=va[0], val_acc=va[1], val_f1=va[2]))
        if verbose:
            print(f"[{tag}|{ep:03d}] tr_loss={tr[0]:.4f} tr_f1={tr[2]:.4f} | val_loss={va[0]:.4f} val_acc={va[1]:.4f} val_f1={va[2]:.4f}")

    # stage 1: frozen backbone, train head only
    set_backbone_trainable(m, False)
    opt1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, m.parameters()),
                             lr=cfg["lr_stage1"], weight_decay=cfg["weight_decay"])
    ep = 0
    for _ in range(stage1_epochs):
        tr = _run_one_epoch(m, tr_ld, crit, opt1, scaler)
        va = _run_one_epoch(m, va_ld, crit)
        _record(ep, 1, tr, va); ep += 1

    # stage 2: full fine-tune
    set_backbone_trainable(m, True)
    opt2 = torch.optim.AdamW(m.parameters(), lr=cfg["lr_stage2"], weight_decay=cfg["weight_decay"])
    if cfg["scheduler"] == "cosine":
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=max(stage2_epochs, 1), eta_min=cfg["lr_stage2"] * 0.01)
    else:
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode="max", factor=0.5, patience=8)
    no_improve = 0
    for _ in range(stage2_epochs):
        tr = _run_one_epoch(m, tr_ld, crit, opt2, scaler)
        va = _run_one_epoch(m, va_ld, crit)
        sched.step(va[2]) if cfg["scheduler"] == "plateau" else sched.step()
        _record(ep, 2, tr, va)
        if va[2] > best["f1"]:
            best.update(f1=va[2], acc=va[1], epoch=ep, state={k: v.detach().cpu().clone() for k, v in m.state_dict().items()})
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
        ep += 1

    res = dict(val_f1=float(best["f1"]), val_acc=float(best["acc"]), best_epoch=int(best["epoch"]),
               epochs_run=len(hist), train_time_min=round((time.time() - t_start) / 60, 2),
               final_train_f1=float(hist[-1]["train_f1"]))
    if keep_model:
        m.load_state_dict(best["state"])
        res.update(model=m, history=hist, test_loader=te_ld, test_ds=te_ds, val_loader=va_ld)
    else:
        del m, opt1, opt2, tr_ld, va_ld, te_ld
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return res

In [ ]:
# ---------------- run the ablation (resumable) ----------------
_cache = {}
ablation_rows = []
if RESUME_ABLATION and os.path.exists(ABLATION_CSV):
    _old = pd.read_csv(ABLATION_CSV)
    ablation_rows = _old.to_dict("records")
    _cache = {r["cfg_json"]: {k: r[k] for k in ("val_f1", "val_acc", "best_epoch", "epochs_run", "train_time_min", "final_train_f1")}
              for r in ablation_rows}
    print(f"Resuming: {len(_cache)} finished runs loaded from {ABLATION_CSV}")


def run_cfg(cfg, factor, value):
    key = json.dumps(cfg, sort_keys=True)
    if key in _cache:
        return _cache[key]
    print(f"\n>>> run {len(_cache) + 1}: {factor} = {value}   cfg={cfg}")
    res = train_with_config(cfg, ABL_STAGE1, ABL_STAGE2, ABL_PATIENCE, tag=f"{factor}={value}")
    _cache[key] = res
    ablation_rows.append({"run": len(_cache), "factor": factor, "value": str(value), **cfg, **res, "cfg_json": key})
    pd.DataFrame(ablation_rows).to_csv(ABLATION_CSV, index=False)     # checkpoint progress after every run
    print(f"    val_f1={res['val_f1']:.4f}  val_acc={res['val_acc']:.4f}  best_epoch={res['best_epoch']}  ({res['train_time_min']} min)")
    return res


best_cfg = dict(BASE_CFG)
best_res = run_cfg(best_cfg, "baseline", "baseline")
print(f"\nBaseline (short budget): val_f1={best_res['val_f1']:.4f}")

decisions, sweep_records = [], []
for factor, values in SEARCH_SPACE.items():
    trial = {}
    for v in values:
        trial[v] = run_cfg({**best_cfg, factor: v}, factor, v)
        sweep_records.append({"factor": factor, "value": str(v), "val_f1": trial[v]["val_f1"], "val_acc": trial[v]["val_acc"],
                              "is_incumbent": v == best_cfg[factor]})
    winner = max(trial, key=lambda v: trial[v]["val_f1"])
    gain = trial[winner]["val_f1"] - best_res["val_f1"]
    if winner != best_cfg[factor] and gain >= MIN_GAIN:
        best_cfg[factor] = winner
        best_res = trial[winner]
        verdict = f"CHANGED -> {winner}  (+{gain:.4f} val macro-F1)"
    else:
        verdict = f"kept {best_cfg[factor]}  (best alternative gain {gain:+.4f} < {MIN_GAIN})"
    decisions.append({"factor": factor, "chosen": best_cfg[factor], "verdict": verdict, "val_f1_after": best_res["val_f1"]})
    print(f"=== {factor}: {verdict}")

print("\nBest config found:")
print(json.dumps(best_cfg, indent=2))
with open("best_config.json", "w") as f:
    json.dump(best_cfg, f, indent=2)
pd.DataFrame(sweep_records).to_csv("ablation_per_factor.csv", index=False)
display(pd.DataFrame(decisions))

In [ ]:
# ---------------- ablation summary: effect of each factor on validation macro-F1 ----------------
abl_df = pd.read_csv(ABLATION_CSV)
show_cols = ["run", "factor", "value", "val_f1", "val_acc", "best_epoch", "epochs_run", "train_time_min"]
display(abl_df[show_cols].sort_values("val_f1", ascending=False).reset_index(drop=True))

sweep_df = pd.DataFrame(sweep_records)       # every value tried per factor (incumbent value included)
factors = list(dict.fromkeys(sweep_df.factor))
ncols = 3
nrows_p = int(np.ceil(len(factors) / ncols))
fig, axes = plt.subplots(nrows_p, ncols, figsize=(5 * ncols, 3.4 * nrows_p))
axes = np.atleast_1d(axes).ravel()
for ax in axes:
    ax.axis("off")
base_f1 = abl_df.loc[abl_df.factor == "baseline", "val_f1"].iloc[0]
for ax, f in zip(axes, factors):
    ax.axis("on")
    sub = sweep_df[sweep_df.factor == f]
    ax.bar(sub.value, sub.val_f1, color=["#F58518" if inc else "#4C78A8" for inc in sub.is_incumbent])
    ax.axhline(base_f1, color="crimson", ls="--", lw=1, label="baseline (short budget)")
    ax.set_ylim(max(0, min(sub.val_f1.min(), base_f1) - 0.05), min(1.0, max(sub.val_f1.max(), base_f1) + 0.02))
    ax.set_title(f + "  (orange = chosen)"); ax.set_ylabel("val macro-F1"); ax.tick_params(axis="x", rotation=30)
axes[0].legend(fontsize=8)
plt.suptitle("Ablation: validation macro-F1 per hyperparameter value (others at current-best)", y=1.01)
plt.tight_layout()
plt.savefig("ablation_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------- final retrain: best config, FULL budget, evaluated once on the test set ----------------
print("Retraining with best config:", best_cfg)
final = train_with_config(best_cfg, STAGE1_EPOCHS, STAGE2_MAX_EPOCHS, PATIENCE, tag="final", verbose=True, keep_model=True)
tuned_model = final["model"]
TUNED_CKPT = f"best_{MODEL_NAME}_tuned.pth"
torch.save(tuned_model.state_dict(), TUNED_CKPT)
pd.DataFrame(final["history"]).to_csv(f"history_{MODEL_NAME}_tuned.csv", index=False)

# the test loader/dataset use the tuned image size & batch size
t_logits, t_true = get_test_predictions(tuned_model, final["test_loader"])
t_probs = torch.softmax(torch.tensor(t_logits), dim=1).numpy()
t_pred = t_probs.argmax(1)

t_acc = accuracy_score(t_true, t_pred)
t_p, t_r, t_f1, _ = precision_recall_fscore_support(t_true, t_pred, average="macro", zero_division=0)
t_kappa = cohen_kappa_score(t_true, t_pred, weights="quadratic")
try:
    t_auc = roc_auc_score(t_true, t_probs, multi_class="ovr", average="macro")
except ValueError:
    t_auc = float("nan")

print(classification_report(t_true, t_pred, target_names=CLASS_NAMES, zero_division=0))

tuned_test = {"accuracy": t_acc, "precision_macro": t_p, "recall_macro": t_r, "f1_macro": t_f1,
              "kappa_quadratic": t_kappa, "auc_roc_ovr_macro": t_auc}
cmp_df = pd.DataFrame({"baseline": baseline_test, "tuned": tuned_test})
cmp_df["change"] = cmp_df["tuned"] - cmp_df["baseline"]
cmp_df.to_csv("baseline_vs_tuned.csv")
display(cmp_df.round(4))

# confusion matrix for the tuned model
t_cm = confusion_matrix(t_true, t_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(t_cm, annot=True, fmt="d", cmap="Greens", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"{MODEL_NAME} (tuned) -- Confusion Matrix (test set)")
plt.tight_layout(); plt.savefig(f"confusion_matrix_{MODEL_NAME}_tuned.png", dpi=150); plt.show()

# misclassified images of the tuned model (same export as Section 9b)
tuned_mis_df, _, _ = export_misclassified(final["test_ds"], t_true, t_pred, t_probs,
                                          out_dir=f"misclassified_{MODEL_NAME}_tuned", split_name="test")
print(f"\nBaseline errors: {len(mis_df)}  ->  tuned errors: {len(tuned_mis_df)}")

# Images wrongly classified by BOTH models are the best candidates for label review
both = set(mis_df.source_path) & set(tuned_mis_df.source_path)
print(f"{len(both)} images are misclassified by both models (check these for label noise).")
pd.DataFrame({"source_path": sorted(both)}).to_csv("misclassified_by_both_models.csv", index=False)

### Reading the ablation results

- **`ablation_results.csv`** lists every run and its validation macro-F1, so you can see which hyperparameters moved the score and which didn't.
- **`baseline_vs_tuned.csv`** compares the baseline and the tuned model on the test set — the only time the test set is used for this section.
- If the tuned test accuracy does not improve over the baseline, the gain in validation was probably noise. In that case, widen the search (more values for `lr_stage2`, image size 320) or try several seeds.
- Images listed in `misclassified_by_both_models.csv` are wrong under both configurations. They are often mislabeled or sit on the boundary between two severity grades, and are worth reviewing manually.

---
### Notes for the other models

Only **Section 4** (`build_model`, `set_backbone_trainable`), `MODEL_NAME`, `IMG_SIZE` and the ablation `build_model_cfg` change per model — everything else (split, augmentation, imbalance handling, training loop, metrics, efficiency measurement, results schema) is identical, so `results_master.csv` stays a fair comparison. Reuse the **same `split.csv`** on every system.

Xception-specific:
- Loaded from **timm** (`legacy_xception`; falls back to `xception` on old timm versions) because torchvision has no Xception. Needs internet on first run to download the weights.
- `IMG_SIZE = 299` (native resolution). The ablation image-size sweep uses `[299, 352]`.
- **Normalisation:** these weights were trained on inputs scaled to [-1, 1], so `NORM_MEAN = NORM_STD = 0.5` is used instead of the ImageNet mean/std. Set `USE_XCEPTION_NORM = False` in Section 3 if you need the exact same normalisation as the other models.
- The head is `fc` and the backbone-freeze helper keys on the `fc.` prefix.
